Дорогой студент!

В домашнем задании Lite вам предлагается поработать подробнее с параметрами словаря и формированием гиперпараметров нейронной сети. Создайте 9 нейросетей с различными гиперпараметрами (см. пунтк 2 и 3)

 Для этого необходимо:

  1. Воссоздать ноутбук, аналогичный ноутбуку практической части №1, загрузив при этом необходимую нам базу (код уже доступен в ноутбуке).

  2. Задать в ноутбуке следующие параметры для размера словаря, ширины окна и шага:

    - Размер словаря - от 10000 до 20000 (выбрать меньшее значение диапазона, если будет перегрузка ОЗУ и перезапуск подключения к Colaboratory)
    - Ширина окна - от 1000 до 2000
    - Шаг - от 100 до 500 (на обучение лучше влияет наименьший шаг, но это может перегрузить ОЗУ).

  3. Создать архитектуру сети и задать гиперпараметры. Можно воспользоваться шаблоном:
  
   - Добавьте модель прямого распространения **Sequential()**
   - Добавьте один или несколько полносвязных (**Dense**) слоёв
   - Добавьте слои **Dropout()** и **BatchNormalization()**
   - Добавьте выходной полносвязный слой с количеством нейронов, соответствующим количеству классов (число писателей)
  
   Напомним, что точность сети можно проверить по значению показателя 'val_accuracy' на конце каждой эпохи.
   

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from tensorflow.keras.preprocessing.text import Tokenizer

from sklearn.metrics import confusion_matrix
import gdown, os, re

In [ ]:
gdown.download('https://storage.yandexcloud.net/aiueducation/Content/base/l7/writers.zip', None, quiet=True)
!unzip -qo writers.zip -d writers/

FILE_DIR  = 'writers'
SIG_TRAIN = 'обучающая'
SIG_TEST  = 'тестовая'

читаем файлы из папки, извлекаем класс и тип (train/test) из имени,
группируем тексты по классам и объединяем их в два списка: text_train и text_test

In [ ]:
CLASS_LIST = []
text_train = []
text_test = []

for file_name in os.listdir(FILE_DIR):
    m = re.match(r'\((.+)\) (\S+)_', file_name)
    if m:
        class_name = m[1]
        subset_name = m[2].lower()

        is_train = SIG_TRAIN in subset_name
        is_test = SIG_TEST in subset_name

        if is_train or is_test:
            if class_name not in CLASS_LIST:
                CLASS_LIST.append(class_name)
                text_train.append('')
                text_test.append('')

            cls = CLASS_LIST.index(class_name)

            with open(f'{FILE_DIR}/{file_name}', 'r') as f:
                text = f.read()

            if is_train:
                text_train[cls] += ' ' + text.replace('\n', ' ')
            else:
                text_test[cls] += ' ' + text.replace('\n', ' ')

CLASS_COUNT = len(CLASS_LIST)

In [ ]:
VOCAB_SIZES = [5000, 10000, 15000]
WIN_SIZES   = [1000, 1500, 2000]
WIN_HOP = 100

Создаем модель

In [ ]:
class DenseNet(nn.Module):
    def __init__(self, input_dim, class_count):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.BatchNorm1d(256),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, class_count)
        )

    def forward(self, x):
        return self.net(x)


In [ ]:
import numpy as np
import torch.optim as optim
from tensorflow.keras.preprocessing.text import Tokenizer

class TextClassifierTorch:
    def __init__(self, text_train, text_test, class_count):
        self.text_train = text_train
        self.text_test = text_test
        self.class_count = class_count

    # 1. токенизация
    def tokenize(self, vocab_size):
        # Ограничиваем словарь (важно для памяти и качества)
        self.tokenizer = Tokenizer(num_words=vocab_size, oov_token='UNK')
          # Строим словарь по обучающей выборке
        self.tokenizer.fit_on_texts(self.text_train)

        # Преобразуем тексты в последовательности индексов слов
        self.seq_train = self.tokenizer.texts_to_sequences(self.text_train)
        self.seq_test  = self.tokenizer.texts_to_sequences(self.text_test)

    # 2. окна
    def split_sequence(self, sequence, win_size, hop):
      # Делим длинный текст на окна фиксированной длины
        return [
            sequence[i:i + win_size]
            for i in range(0, len(sequence) - win_size + 1, hop)
        ]

    # 3. формирование выборок (y — индексы классов)
    def vectorize(self, seq_list, win_size, hop):
        x, y = [], []
        for cls in range(len(seq_list)):
            vectors = self.split_sequence(seq_list[cls], win_size, hop)
            x += vectors
            y += [cls] * len(vectors)
        return x, np.array(y)

    # 4. подготовка данных
    def prepare_data(self, vocab_size, win_size, hop):
        x_train, y_train = self.vectorize(self.seq_train, win_size, hop)
        x_test,  y_test  = self.vectorize(self.seq_test, win_size, hop)

        # BoW
        x_train = self.tokenizer.sequences_to_matrix(x_train)
        x_test  = self.tokenizer.sequences_to_matrix(x_test)

        # Перевод в torch-тензоры
        self.x_train = torch.tensor(x_train, dtype=torch.float32)
        self.y_train = torch.tensor(y_train, dtype=torch.long)

        self.x_test = torch.tensor(x_test, dtype=torch.float32)
        self.y_test = torch.tensor(y_test, dtype=torch.long)

    # 5. обучение
    def train(self, vocab_size, win_size, hop, epochs=5, batch_size=128):
        self.tokenize(vocab_size)
        self.prepare_data(vocab_size, win_size, hop)

        model = DenseNet(vocab_size, self.class_count)

        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=1e-3)

        for epoch in range(epochs):
            model.train()
            # Перемешивание данных
            perm = torch.randperm(self.x_train.size(0))

            for i in range(0, self.x_train.size(0), batch_size):
                idx = perm[i:i+batch_size]

                xb = self.x_train[idx]
                yb = self.y_train[idx]

                optimizer.zero_grad()
                outputs = model(xb)
                loss = criterion(outputs, yb)
                loss.backward()
                optimizer.step() # обновление весов

        # валидация
        model.eval()
        with torch.no_grad():
            outputs = model(self.x_test)
            preds = torch.argmax(outputs, dim=1)
            acc = (preds == self.y_test).float().mean().item()

        return acc

In [ ]:

classifier = TextClassifierTorch(text_train, text_test, CLASS_COUNT, device)

results = []

for vocab in [10000, 15000, 20000]:
    for win in [1000, 1500, 2000]:
        acc = classifier.train(vocab, win, 100)

        results.append({
            "vocab": vocab,
            "win": win,
            "hop": 100,
            "val_acc": acc
        })

        print(f"VOCAB={vocab}, WIN={win} → {acc:.4f}")

results = sorted(results, key=lambda x: x["val_acc"], reverse=True)

VOCAB=10000, WIN=1000 → 0.9284
VOCAB=10000, WIN=1500 → 0.9478
VOCAB=10000, WIN=2000 → 0.9438
VOCAB=15000, WIN=1000 → 0.9219
VOCAB=15000, WIN=1500 → 0.9369
VOCAB=15000, WIN=2000 → 0.9479
VOCAB=20000, WIN=1000 → 0.9314
VOCAB=20000, WIN=1500 → 0.9447
VOCAB=20000, WIN=2000 → 0.9488
